<a href="https://colab.research.google.com/github/Venomm777/FlyAi/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Venomm777/FlyAi/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type: Ranking / Priority Scoring (Learning-to-Rank via Binary Classification Probabilities)**

* **Why:** The end operational goal is not simply to flag pages as "declining" or "healthy" in isolation, but to deliver an ordered weekly content refresh queue. Editorial bandwidth is strictly capped (an editorial team can typically only update 20 to 50 URLs per week).
* **Framing:** We frame this as a probability-scoring problem. The model outputs a continuous decay probability score $\hat{p} \in [0, 1]$ for every candidate page, allowing us to sort all site pages from highest to lowest risk of traffic loss.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
import pandas as pd
import numpy as np

# Load data and check target distribution
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

declining_count = df["is_declining_label"].sum()
total_count = len(df)
base_rate = df["is_declining_label"].mean()

print(f"Total Pages Analyzed: {total_count:,}")
print(f"Declining Pages:      {declining_count:,}")
print(f"Baseline Class Balance (Declining Rate): {base_rate:.3%}")

## 3. Success metric

*One metric you can defend. What number means 'good'?*
**Primary Success Metric: Precision@K (specifically Precision@20 and Precision@50)**

* **Defense:** An SEO team does not review all 30,000 pages; they review the top $K$ items in the ranked queue each cycle. Global metrics like ROC-AUC or Log-Loss evaluate the entire distribution, whereas Precision@50 directly measures queue utility: out of the top 50 pages recommended for an update, what proportion actually suffered performance decline?
* **What 'Good' Means:** A hand-written baseline rule achieves ~0.240 Precision@50 on client holdout data. A "good" ML model should achieve a Precision@50 $\ge 0.650$ (a $>2.5\times$ lift over the heuristic rule), ensuring that over two-thirds of an editor's manual review hours are spent on truly decaying assets.

In [ ]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(topk.mean())

# Verify metric behavior against dummy/random baseline
y_true = df["is_declining_label"].values
random_scores = np.random.RandomState(42).rand(len(y_true))

print(f"Random Baseline Precision@20: {precision_at_k(random_scores, y_true, 20):.3f}")
print(f"Random Baseline Precision@50: {precision_at_k(random_scores, y_true, 50):.3f}")
print(f"Target Benchmark for ML Model Precision@50: >= 0.650")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis: One row = One unique URL / Content Page**

Each row summarizes a single page's performance across a rolling 90-day observation window within a specific client domain. The features capture historical traffic, search visibility, keyword rank depth, and content characteristics.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why Fixed If-Else Rules Fail:**
1. **Multi-Factor Interaction:** A static rule such as `if days_since_last_update > 180 and impressions > 1000` is too rigid. High-authority evergreen pages can remain stable for two years without updates, whereas dynamic product or comparison pages can experience severe CTR decay in 60 days.
2. **Threshold Cliffs:** A fixed rule treats a page updated 179 days ago as completely healthy, while flagging a page updated 181 days ago. Machine learning models generate smooth, non-linear scoring surfaces across position, CTR, and age.
3. **Severe Score Ties:** A Boolean rule divides pages into broad buckets, creating massive ties at the top of the queue. A learned model produces granular continuous risk scores, enabling rank ordering.

In [ ]:
unit_cols = [
    "client_id" if "client_id" in df.columns else "site_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "is_declining_label"
]
available_cols = [c for c in unit_cols if c in df.columns]

unit_df = df[available_cols].head(5)
print(f"Shape: {df[available_cols].shape[0]} rows x {df[available_cols].shape[1]} columns")
unit_df

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.